# 🍓 Strawberry Classifier

Complete YOLO11 workflow for strawberry detection and classification.

**Installation:** `pip install -e .`

## Setup

In [2]:
from pathlib import Path
from strawberry_classifier import StrawberryDetector, StrawberryClassifier
from strawberry_classifier.data.prepare import prepare_dataset
from strawberry_classifier.data.crop_extractor import CropExtractor
from strawberry_classifier.data.utils import validate_dataset_structure

DATA_DIR = Path("data")
SOURCE_DATASET = "farm_dataset"

## Data Preparation

In [8]:
source_dir = DATA_DIR / "raw" / SOURCE_DATASET
output_dir = DATA_DIR / "processed" / SOURCE_DATASET

if source_dir.exists():
    prepare_dataset(
        source_dir=source_dir,
        output_dir=output_dir,
        split_ratios=(0.7, 0.2, 0.1),
        seed=42
    )
    
    stats = validate_dataset_structure(output_dir)
    for split in ['train', 'val', 'test']:
        images = stats.get(f'{split}_images', 0)
        labels = stats.get(f'{split}_labels', 0)
        print(f"{split}: {images} images, {labels} labels")
else:
    print(f"Source dataset not found: {source_dir}")

train: 457 images, 457 labels
val: 130 images, 130 labels
test: 67 images, 67 labels


## Crop Extraction

In [3]:
dataset_path = DATA_DIR / "processed" / SOURCE_DATASET
classification_output = DATA_DIR / "classification" / SOURCE_DATASET

if dataset_path.exists():
    extractor = CropExtractor(
        dataset_path=dataset_path,
        output_dir=classification_output,
        crop_size=(224, 224),
        padding=0.1,
        preserve_splits=True
    )
    
    extractor.extract_crops()
    stats = extractor.get_stats()
    
    print(f"Total crops: {stats['total_crops']}")
    for class_name, count in stats['crops_per_class'].items():
        print(f"{class_name}: {count}")
else:
    print("Run data preparation first")

Total crops: 4612
Flower: 756
Flower_Fruit: 985
Green_Fruit: 952
Green_White_Fruit: 838
Red_Fruit: 625
UNSURE: 7
White_Red_Fruit: 449


## Detection Training

In [4]:
from pathlib import Path

dataset_path = Path(DATA_DIR) / "processed" / SOURCE_DATASET
data_yaml = dataset_path / "data.yaml"

if data_yaml.exists():
    detector = StrawberryDetector("yolo11n.pt")
    
    # Training with minimal epochs for testing
    results = detector.train(
        data=data_yaml,
        epochs=2,  # Very small for testing
        batch=4,   # Small batch size
        imgsz=640,
        project="runs/train",
        name="detection_test"
    )
    
    BEST_DETECTION_MODEL = f"{results.save_dir}/weights/best.pt"
    print(f"Model saved: {BEST_DETECTION_MODEL}")
else:
    print("Dataset not prepared")

Ultralytics 8.3.169 🚀 Python-3.12.3 torch-2.7.1+cu126 CPU (Intel Core(TM) i5-8365U 1.60GHz)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/processed/farm_dataset/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=2, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=detection_test, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=Tru

train: Scanning /home/fabian/strawberry_classifier/data/processed/farm_dataset/train/labels.cache... 457 images, 2 backgrounds, 0 corrupt: 100%|██████████| 457/457 [00:00<?, ?it/s]

train: /home/fabian/strawberry_classifier/data/processed/farm_dataset/train/images/4cc52e98-IMG_0130.JPEG: 1 duplicate labels removed
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1469.5±602.7 MB/s, size: 724.1 KB)



/home/fabian/strawberry_classifier/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
val: Scanning /home/fabian/strawberry_classifier/data/processed/farm_dataset/val/labels.cache... 130 images, 0 backgrounds, 0 corrupt: 100%|██████████| 130/130 [00:00<?, ?it/s]
/home/fabian/strawberry_classifier/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Plotting labels to runs/train/detection_test/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.000909, momentum=0.9) with parameter groups 81 weight(decay=0.0), 88 weight(decay=0.0005), 87 bias(decay=0.0)


2025/07/25 16:01:08 INFO mlflow.tracking.fluent: Experiment with name 'runs/train' does not exist. Creating a new experiment.


MLflow: logging run_id(f13381960dd84f42a53e0da84e4d611c) to /home/fabian/strawberry_classifier/runs/mlflow
MLflow: view at http://127.0.0.1:5000 with 'mlflow server --backend-store-uri /home/fabian/strawberry_classifier/runs/mlflow'
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to runs/train/detection_test
Starting training for 2 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


        1/2         0G      2.111      4.396       1.52         87        640:   3%|▎         | 3/115 [00:11<07:19,  3.93s/it]


KeyboardInterrupt: 

## Classification Training

In [5]:
if classification_output.exists():
    classifier = StrawberryClassifier("yolo11n-cls.pt")
    
    results = classifier.train(
        data=classification_output,
        epochs=50,
        batch=32,
        imgsz=224,
        project="runs/train",
        name="classification_experiment"
    )
    
    BEST_CLASSIFICATION_MODEL = f"{results.save_dir}/weights/best.pt"
    print(f"Model saved: {BEST_CLASSIFICATION_MODEL}")
else:
    print("Run crop extraction first")

100%|██████████| 5.52M/5.52M [00:00<00:00, 7.19MB/s]


Ultralytics 8.3.169 🚀 Python-3.12.3 torch-2.7.1+cu126 CPU (Intel Core(TM) i5-8365U 1.60GHz)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/classification/farm_dataset, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=classification_experiment, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.

train: Scanning /home/fabian/strawberry_classifier/data/classification/farm_dataset/train... 3234 images, 0 corrupt: 100%|██████████| 3234/3234 [00:05<00:00, 601.93it/s]


train: New cache created: /home/fabian/strawberry_classifier/data/classification/farm_dataset/train.cache
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 441.6±124.9 MB/s, size: 16.5 KB)


/home/fabian/strawberry_classifier/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
val: Scanning /home/fabian/strawberry_classifier/data/classification/farm_dataset/val... 891 images, 0 corrupt: 100%|██████████| 891/891 [00:01<00:00, 579.10it/s]

val: New cache created: /home/fabian/strawberry_classifier/data/classification/farm_dataset/val.cache



/home/fabian/strawberry_classifier/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.000909, momentum=0.9) with parameter groups 39 weight(decay=0.0), 40 weight(decay=0.0005), 40 bias(decay=0.0)
MLflow: logging run_id(f13381960dd84f42a53e0da84e4d611c) to /home/fabian/strawberry_classifier/runs/mlflow
MLflow: view at http://127.0.0.1:5000 with 'mlflow server --backend-store-uri /home/fabian/strawberry_classifier/runs/mlflow'
MLflow: disable with 'yolo settings mlflow=False'
WARNING ⚠️ MLflow: Failed to initialize: Changing param values is not allowed. Param with key='task' was already logged with value='detect' for run ID='f13381960dd84f42a53e0da84e4d611c'. Attempted logging new value 'classify'.
WARNING ⚠️ MLflow: Not tracking this run
Image sizes 224 train, 224 val
Using 0 dataloader workers
Logging results to runs/train/classification_experiment
Starting training for 50 epochs...

      Epoch    GPU

       1/50         0G      2.165         32        224:   2%|▏         | 2/102 [00:04<03:50,  2.31s/it]


KeyboardInterrupt: 

## Validation

In [6]:
if 'BEST_DETECTION_MODEL' in locals():
    detector = StrawberryDetector(BEST_DETECTION_MODEL)
    results = detector.validate(data=data_yaml)
    
    if hasattr(results, 'box'):
        print(f"mAP50: {results.box.map50:.4f}")
        print(f"mAP50-95: {results.box.map:.4f}")
else:
    print("No detection model to validate")

No detection model to validate


In [ ]:
if 'BEST_CLASSIFICATION_MODEL' in locals():
    classifier = StrawberryClassifier(BEST_CLASSIFICATION_MODEL)
    results = classifier.validate(data=classification_data_yaml)
    
    if hasattr(results, 'top1'):
        print(f"Top1 Accuracy: {results.top1:.4f}")
        print(f"Top5 Accuracy: {results.top5:.4f}")
else:
    print("No classification model to validate")

## Predictions

In [ ]:
image_path = "data/processed/farm_dataset/test/images/image_001.jpg"

if 'BEST_DETECTION_MODEL' in locals():
    detector = StrawberryDetector(BEST_DETECTION_MODEL)
    results = detector.predict(image_path, save=True)
    print(f"Detection results saved")

if 'BEST_CLASSIFICATION_MODEL' in locals():
    classifier = StrawberryClassifier(BEST_CLASSIFICATION_MODEL)
    results = classifier.predict(image_path, save=True)
    print(f"Classification results saved")

## Summary

Complete workflow for strawberry detection and classification using YOLO11.

## Dataset Utilities

Advanced dataset operations using integrated utilities.

In [8]:
from strawberry_classifier.data import validate_dataset, analyze_dataset

# Validate dataset structure and format
dataset_path = "data/processed/farm_dataset"
validation_result = validate_dataset(dataset_path)

print(f"Dataset valid: {validation_result['valid']}")
print(f"Total issues: {validation_result['total_issues']}")

if validation_result['structure_issues']:
    print("\nStructure Issues:")
    for issue in validation_result['structure_issues']:
        print(f"  ❌ {issue}")

if validation_result['images_issues']:
    print("\nImage/Label Issues:")
    for issue in validation_result['images_issues'][:5]:
        print(f"  ⚠️  {issue}")

stats = validation_result['stats']
print(f"\nDataset Statistics:")
print(f"  Images: {stats['images']}")
print(f"  Labels: {stats['labels']}")
print(f"  Orphaned images: {stats['orphaned_images']}")
print(f"  Orphaned labels: {stats['orphaned_labels']}")

Dataset valid: False
Total issues: 3

Image/Label Issues:
  ⚠️  train: 454 labels without images
  ⚠️  val: 127 labels without images
  ⚠️  test: 66 labels without images

Dataset Statistics:
  Images: 7
  Labels: 654
  Orphaned images: 0
  Orphaned labels: 647


In [9]:
# Analyze dataset label distribution
analysis = analyze_dataset(dataset_path)

print(f"Dataset Analysis: {dataset_path}")
print("=" * 50)

for split_name, counts in analysis['split_counts'].items():
    if not counts:
        continue
    
    print(f"\n{split_name.upper()} Split:")
    print("-" * 20)
    
    total_labels = sum(counts.values())
    print(f"Total labels: {total_labels}")
    
    for class_id in sorted(counts.keys()):
        class_name = analysis['class_names'][class_id] if class_id < len(analysis['class_names']) else f"Class_{class_id}"
        count = counts[class_id]
        percentage = (count / total_labels) * 100
        print(f"  {class_name}: {count} ({percentage:.1f}%)")

Dataset Analysis: data/processed/farm_dataset

TRAIN Split:
--------------------
Total labels: 3306
  Flower: 562 (17.0%)
  Flower_Fruit: 745 (22.5%)
  Green_Fruit: 679 (20.5%)
  Green_White_Fruit: 588 (17.8%)
  Red_Fruit: 425 (12.9%)
  UNSURE: 5 (0.2%)
  White_Red_Fruit: 302 (9.1%)

VAL Split:
--------------------
Total labels: 915
  Flower: 133 (14.5%)
  Flower_Fruit: 181 (19.8%)
  Green_Fruit: 203 (22.2%)
  Green_White_Fruit: 181 (19.8%)
  Red_Fruit: 125 (13.7%)
  UNSURE: 2 (0.2%)
  White_Red_Fruit: 90 (9.8%)

TEST Split:
--------------------
Total labels: 489
  Flower: 74 (15.1%)
  Flower_Fruit: 124 (25.4%)
  Green_Fruit: 81 (16.6%)
  Green_White_Fruit: 77 (15.7%)
  Red_Fruit: 76 (15.5%)
  White_Red_Fruit: 57 (11.7%)


In [3]:
from strawberry_classifier.data import compare_datasets

# Compare two datasets for overlap (example)
# dataset1_path = "data/processed/farm_dataset"
# dataset2_path = "data/processed/another_dataset"

# Uncomment to compare actual datasets:
# comparison_results = compare_datasets(dataset1_path, dataset2_path)
# 
# print(f"\nDataset Comparison:")
# print("=" * 60)
# 
# for split1, split_results in comparison_results.items():
#     print(f"\n{split1.upper()} split:")
#     print("-" * 30)
#     
#     for split2, stats in split_results.items():
#         print(f"  vs {split2}: {stats['overlap_count']}/{stats['dataset1_count']} "
#               f"({stats['overlap_percentage']:.1f}%) images overlap")

print("Dataset comparison ready - uncomment code above to use with actual datasets")

Dataset comparison ready - uncomment code above to use with actual datasets


### Dataset Operations

In [4]:
from strawberry_classifier.data import merge_datasets, create_subset

# Merge two datasets (example)
# dataset1_path = "data/processed/farm_dataset"
# dataset2_path = "data/processed/another_dataset"
# output_path = "data/processed/merged_dataset"

# Uncomment to merge actual datasets:
# merge_stats = merge_datasets(
#     dataset1_path=dataset1_path,
#     dataset2_path=dataset2_path,
#     output_path=output_path,
#     dataset1_name="farm",
#     dataset2_name="additional"
# )
# 
# print(f"Merge complete:")
# print(f"  Total images: {merge_stats['images']}")
# print(f"  Total labels: {merge_stats['labels']}")
# print(f"  Classes: {merge_stats['classes']}")

print("Dataset merging ready - uncomment code above to use")

Dataset merging ready - uncomment code above to use


In [5]:
# Create dataset subset for quick testing
input_dataset = "data/processed/farm_dataset"
subset_output = "data/processed/farm_dataset_subset"

# Uncomment to create actual subset:
# subset_stats = create_subset(
#     dataset_path=input_dataset,
#     output_path=subset_output,
#     subset_ratio=0.1,  # 10% of original dataset
#     random_seed=42
# )
# 
# print(f"Subset created:")
# print(f"  Original: {subset_stats['total']} images")
# print(f"  Subset: {subset_stats['selected']} images (10.0%)")

print("Subset creation ready - uncomment code above to create 10% subset")

Subset creation ready - uncomment code above to create 10% subset


### Color Transfer

In [6]:
from strawberry_classifier.data import apply_color_transfer_to_dataset

# Apply color transfer for data augmentation
source_dataset = "data/processed/farm_dataset"
reference_images = "data/reference_lighting"  # Directory with reference images
output_dataset = "data/processed/farm_dataset_color_transferred"

# Uncomment to apply color transfer:
# color_stats = apply_color_transfer_to_dataset(
#     dataset_path=source_dataset,
#     source_reference=reference_images,
#     output_path=output_dataset,
#     probability=0.5,  # Apply to 50% of images
#     intensity=1.0     # Full intensity
# )
# 
# print(f"Color transfer complete:")
# print(f"  Processed: {color_stats['processed']} images")
# print(f"  Skipped: {color_stats['skipped']} images")

print("Color transfer ready - prepare reference images and uncomment code above")

Color transfer ready - prepare reference images and uncomment code above


### CLI Usage Examples

All dataset utilities are also available through the CLI:

In [ ]:
# CLI commands for dataset utilities:

print("Available CLI commands:")
print()
print("# Dataset validation:")
print("python -m strawberry_classifier validate-dataset --dataset data/processed/farm_dataset")
print()
print("# Dataset analysis:")
print("python -m strawberry_classifier analyze-dataset --dataset data/processed/farm_dataset")
print()
print("# Compare datasets:")
print("python -m strawberry_classifier compare-datasets --dataset1 data/dataset1 --dataset2 data/dataset2")
print()
print("# Merge datasets:")
print("python -m strawberry_classifier merge-datasets --dataset1 data/dataset1 --dataset2 data/dataset2 --output data/merged")
print()
print("# Create subset:")
print("python -m strawberry_classifier create-subset --dataset data/input --output data/subset --ratio 0.1")
print()
print("# Color transfer:")
print("python -m strawberry_classifier color-transfer --dataset data/input --reference reference_dir --output data/output")
print()
print("# View all available commands:")
print("python -m strawberry_classifier --help")

In [ ]:
print("Workflow completed:")
print(f"- Detection model: {'trained' if 'BEST_DETECTION_MODEL' in locals() else 'not trained'}")
print(f"- Classification model: {'trained' if 'BEST_CLASSIFICATION_MODEL' in locals() else 'not trained'}")
print(f"- Data prepared: True")
print(f"- Crops extracted: True")